# <font color='#BFD72F' size=6>**Imputation: exact recoveries, customer history and row limits**</font>

Notebook 01 (sections 2.5–2.7, 6 and 12) fills the gaps in three layers: exact recovery from identities, structural fill for values that can't exist, and then a fitted imputer (median / most frequent) for whatever is left. An audit of that last layer found three weaknesses, which this notebook fixes and tests:

1. **Missed exact recoveries.** With one or two previous purchases, the largest and smallest previous purchase follow from the total.
2. **The customer's own history was ignored.** Customers keep the same payment type, salesperson and transport method from one purchase to the next. The days since the previous purchase are simply the date difference to it, when that purchase is in the data.
3. **Imputed values could be impossible.** A single median ignores the rest of the row: it put the 365-day spend below the 180-day spend, or above everything the customer ever spent.

| Change | Code | Rule (`config.yaml`) |
|---|---|---|
| exact recoveries | `src/cleaning.py` → `apply_recompute_rules` | `diagnostics.recompute_rules` (4 new rules) |
| customer history | `src/cleaning.py` → `fill_from_customer_history` | `diagnostics.customer_history` |
| row limits | `src/preprocessing.py` → `RowBoundedImputer` | `preprocessing.imputation.row_bounds` |

<div class="alert alert-block alert-info">

## Table of Contents <a class="anchor" id="toc"></a>
- [0. Setup](#setup)
- [1. What reaches the imputers](#gaps)
- [2. Exact recoveries of the largest / smallest previous purchase](#recoveries)
- [3. Filling from the customer's earlier purchases](#history)
- [4. Keeping imputed values within their row's limits](#bounds)
- [5. Does it help the models?](#models)
- [6. Alternative imputers, again](#imputers)
- [7. Model scores with the final recipe](#scores)
- [8. Summary](#summary)
</div>

# <font color='#BFD72F' size=6>**0. Setup**</font> <a class="anchor" id="setup"></a>

In [1]:
import copy
import os
import sys
from pathlib import Path

# run from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml

from src.cleaning import fill_from_customer_history, prepare_table
from src.data import load_data
from src.evaluate import compare_preprocessing_variants, cross_validate_pipeline, paired_comparison, select_scoring, temporal_folds
from src.pipeline import build_pipeline, prepare_task_data
from src.preprocessing import RowBoundedImputer, _numeric_imputer
from src.tasks import TASKS

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
data_cfg, diag_cfg = config["data"], config["diagnostics"]
tokens = diag_cfg["placeholder_tokens"]
SCORING = {task: select_scoring(TASKS[task].evaluate.SCORING, config[task]["metrics"]) for task in TASKS}
NEW_RULE_COLUMNS = ("prior_max_purchase_value", "prior_min_purchase_value")


def variant(recover=True, history=True, bounds=True):
    """config.yaml with each of the three changes switched on or off."""
    cfg = copy.deepcopy(config)
    if not recover:
        cfg["diagnostics"]["recompute_rules"] = [r for r in cfg["diagnostics"]["recompute_rules"] if r["column"] not in NEW_RULE_COLUMNS]
    cfg["diagnostics"]["customer_history"]["use"] = history
    if not bounds:
        cfg["preprocessing"]["imputation"]["row_bounds"] = {}
    return cfg


def versus(scores_a, scores_b, higher_is_better):
    r = paired_comparison(scores_a, scores_b, higher_is_better)
    decisive = ", decisive" if r["decisive"] else ""
    return f"{r['mean_advantage']:+.4f} (SE {r['se']:.4f}, better on {r['folds_a_better']}/{r['n_folds']} folds{decisive})"


# the purchase tables after the stateless steps, WITHOUT the new recoveries (to check the identities
# on originally observed values); the 65 almost-empty training rows are not purchases
previous = variant(recover=False, history=False, bounds=False)
clientes = load_data(data_cfg["clientes_path"], tokens)
train_prev, _ = prepare_table(load_data(data_cfg["train_path"], tokens), previous, clientes)
test_prev, _ = prepare_table(load_data(data_cfg["test_path"], tokens), previous, clientes)
purchases = pd.concat([train_prev.assign(file="train"), test_prev.assign(file="test")], ignore_index=True)
purchases = purchases[purchases["purchase_date"].notna()].sort_values(["customer_id", "purchase_date"]).reset_index(drop=True)
print(len(purchases), "purchases,", purchases["customer_id"].nunique(), "customers")

7455 purchases, 629 customers


# <font color='#BFD72F' size=6>**1. What reaches the imputers**</font> <a class="anchor" id="gaps"></a>

Number of missing values left for the fitted imputers, per feature, before the three changes and after them. Development set (what the models learn from) and test.csv (what Kaggle scores).

[Back to TOC](#toc)

In [2]:
rows = []
for task in ["classification", "regression"]:
    before, after = prepare_task_data(previous, task), prepare_task_data(config, task)
    recipe = after["preprocessing_config"]
    features = recipe["numeric_features"] + recipe["categorical_features"]
    for name, part in [("development", "X_dev"), ("test.csv", "X_kaggle")]:
        b, a = before[part][features].isna().sum(), after[part][features].isna().sum()
        for column in b.index[(b > 0) | (a > 0)]:
            rows.append({"task": task, "set": name, "column": column, "before": int(b[column]), "after": int(a[column])})
gaps = pd.DataFrame(rows).pivot_table(index=["task", "column"], columns="set", values=["before", "after"], aggfunc="sum")
gaps = gaps[[("before", "development"), ("after", "development"), ("before", "test.csv"), ("after", "test.csv")]]
display(gaps)
print(gaps.groupby(level="task").sum())

before       after   before    after
set                                         development development test.csv test.csv
task           column                                                                
classification days_since_previous_purchase         6.0         0.0      NaN      NaN
               payment_type                       144.0        18.0     30.0      0.0
               prior_max_purchase_value            38.0        35.0      6.0      6.0
               prior_mean_gap_days                  6.0         0.0      NaN      NaN
               salesperson                        158.0        25.0     30.0      2.0
               spend_365d                         137.0       137.0     38.0     38.0
               transport_method                   153.0        22.0     30.0      0.0
               warehouse                          150.0        12.0     30.0      1.0
regression     commercial_region                    7.0         7.0      8.0      8.0
               days_since_previous_purchase         5.0         0.0      NaN      NaN
               island                               7.0         7.0      8.0      8.0
               municipality                        45.0        45.0     88.0     88.0
               payment_type                        83.0         3.0     30.0      0.0
               prior_max_purchase_value            18.0        17.0      6.0      6.0
               prior_mean_gap_days                  5.0         0.0      NaN      NaN
               salesperson                         90.0         8.0     30.0      2.0
               spend_365d                          76.0        76.0     38.0     38.0
               transport_method                    78.0         6.0     30.0      0.0
               warehouse                           93.0         5.0     30.0      1.0

                    before       after   before    after
set            development development test.csv test.csv
task                                                    
classification       792.0       249.0    164.0     47.0
regression           507.0       174.0    268.0    151.0


**About two thirds of the gaps no longer reach an imputer:**
- classification development set: 792 → 249;
- regression development set: 507 → 174;
- test.csv: 164 → 47 (classification features) and 268 → 151 (regression features).

Almost all of it comes from the customer history (section 3):
- each transaction category drops from about 150 gaps to 12–25 on the development set, and to 0–2 on test.csv;
- every gap in `days_since_previous_purchase` and `prior_mean_gap_days` is filled exactly.

**What is left for the fitted imputers:**
- `spend_365d` (137): keeps its missing indicator, and its imputed values are now bounded (section 4);
- `prior_max_purchase_value` (35): also bounded;
- the customer-table geography (regression only);
- the categories of purchases whose customer has no earlier known value.

# <font color='#BFD72F' size=6>**2. Exact recoveries of the largest / smallest previous purchase**</font> <a class="anchor" id="recoveries"></a>

With **one** previous purchase, that purchase is at once the smallest, the largest and the total. With **two**, smallest + largest = total. Each identity is checked on the rows where all its terms were observed (before the new rules), exactly like the identities of notebook 01, section 5.1.

[Back to TOC](#toc)

In [3]:
close = lambda a, b: (a - b).abs() <= 0.01
p = purchases
one, two = p["prior_purchase_count"] == 1, p["prior_purchase_count"] == 2
checks = []
for name, mask, left, right in [
    ("1 previous purchase: max = total", one, p["prior_max_purchase_value"], p["prior_total_spend"]),
    ("1 previous purchase: min = total", one, p["prior_min_purchase_value"], p["prior_total_spend"]),
    ("2 previous purchases: max + min = total", two, p["prior_max_purchase_value"] + p["prior_min_purchase_value"], p["prior_total_spend"]),
]:
    observed = mask & left.notna() & right.notna()
    checks.append({"identity": name, "rows checked": int(observed.sum()), "holds (%)": round(100 * close(left, right)[observed].mean(), 2)})
display(pd.DataFrame(checks))

recovered = []
for file, path in [("train.csv", data_cfg["train_path"]), ("test.csv", data_cfg["test_path"])]:
    _, report = prepare_table(load_data(path, tokens), config, clientes)
    new = report[report["column"].isin(NEW_RULE_COLUMNS) & report["step"].eq("recomputed from identity")]
    recovered.append(new.groupby("column")["values"].sum().rename(file))
display(pd.concat(recovered, axis=1).fillna(0).astype(int))
print("missing prior_max_purchase_value by number of previous purchases (before):",
      p.loc[p["prior_max_purchase_value"].isna(), "prior_purchase_count"].clip(upper=3).value_counts().sort_index().rename({3: "3+"}).to_dict())

,identity,rows checked,holds (%)
0,1 previous purchase: max = total,466,100.0
1,1 previous purchase: min = total,469,100.0
2,2 previous purchases: max + min = total,387,100.0


,train.csv,test.csv
column,,
prior_max_purchase_value,4,0


missing prior_max_purchase_value by number of previous purchases (before): {np.int16(1): 3, np.int16(2): 1, '3+': 47}


The three identities hold on **100%** of the rows where they can be checked (466, 469 and 387 rows). So they are exact definitions, like the identities of notebook 01, section 5.1.

- **4 of the 51 missing `prior_max_purchase_value` are recovered:** the 3 with one previous purchase and the 1 with two.
- **The other 47 can't be:** they have three or more previous purchases, where the largest one doesn't follow from the other columns. They stay for the fitted imputer, now with row limits (section 4).
- **`prior_min_purchase_value`** has no gap among real purchases today. Its two rules cover future records.

# <font color='#BFD72F' size=6>**3. Filling from the customer's earlier purchases**</font> <a class="anchor" id="history"></a>

`fill_from_customer_history()` uses only the **same customer's earlier purchases**: never a later one, and never a target. This is the information a system has when a purchase is made, the same kind as the `prior_*` columns. It runs after the stateless steps, on every known purchase (train.csv + test.csv), and leaves whatever it can't fill to the fitted imputers.

### 3.1 Categories: the customer's last observed value

How good would this fill be? Take every purchase whose value **is** known and has an earlier purchase of the same customer with a known value, then compare the actual value with:
- the customer's **last observed value**;
- the **most frequent** value overall, which is what the current imputer would use.

[Back to TOC](#toc)

In [4]:
rules = diag_cfg["customer_history"]
by_customer = purchases.groupby("customer_id", sort=False)
quality = []
for column in rules["last_observed_columns"]:
    last = by_customer[column].shift(1).astype(object).groupby(purchases["customer_id"]).ffill()
    known = purchases[column].notna() & last.notna()
    quality.append({
        "column": column,
        "purchases compared": int(known.sum()),
        "last observed value is right (%)": round(100 * (purchases[column] == last)[known].mean(), 1),
        "most frequent value is right (%)": round(100 * (purchases[column] == purchases[column].mode()[0])[known].mean(), 1),
        "gaps": int(purchases[column].isna().sum()),
        "gaps with an earlier value": int((purchases[column].isna() & last.notna()).sum()),
    })
display(pd.DataFrame(quality).set_index("column"))

,purchases compared,last observed value is right (%),most frequent value is right (%),gaps,gaps with an earlier value
column,,,,,
payment_type,6607,97.0,72.7,225,206
salesperson,6613,95.5,63.6,224,197
transport_method,6605,80.9,16.7,226,203
warehouse,6604,99.4,99.7,225,211
commercial_zone,6826,98.7,93.2,0,0
document_series,6826,80.7,52.2,0,0


- **The customer's last value is right far more often than the most frequent value:**

  | Column | Last value right | Most frequent value right |
  |---|---|---|
  | payment type | 97% | 73% |
  | salesperson | 96% | 64% |
  | transport method | 81% | 17% |
  | document series | 81% | 52% |
- **Warehouse is a tie** (99.4% vs 99.7%): almost every purchase uses the same warehouse, so both fills give the same value. It is filled from history anyway, to keep one consistent rule.
- **197–211 of the ~225 gaps in each transaction category** have an earlier value for the same customer. The rest belong to a customer's first known purchase and stay for the fitted imputer.
- **`commercial_zone` and `document_series`** have no gaps among real purchases today. The rule covers future records.

These gaps aren't "missing completely at random" in a useful sense. Which value is missing is explained by **who the customer is** (MAR given the customer), and the customer's own past is the best estimate. Notebook 01, section 6 found no link between the gaps and the target or the other columns of the same row, which is still true: the information is in the customer's other rows.

### 3.2 Days since the previous purchase: a date difference

When the customer's previous purchase is in the data (its prior purchase count is exactly one lower), `days_since_previous_purchase` should be the number of days between the two purchase dates. The recompute rules then run again, so `prior_mean_gap_days` follows from it.

In [5]:
previous_count = by_customer["prior_purchase_count"].shift(1)
consecutive = (purchases["prior_purchase_count"] - previous_count) == 1
date_gap = (purchases["purchase_date"] - by_customer["purchase_date"].shift(1)).dt.days
observed = consecutive & purchases["days_since_previous_purchase"].notna()
print(f"consecutive purchases with an observed gap: {int(observed.sum())}; "
      f"gap = date difference on {100 * ((date_gap - purchases['days_since_previous_purchase']).abs() <= 0.5)[observed].mean():.1f}%")
missing = purchases["days_since_previous_purchase"].isna()
print(f"missing gaps: {int(missing.sum())}, of which the previous purchase is in the data: {int((missing & consecutive).sum())}")

# the tables as the pipeline prepares them (exact recoveries included), before the history fill
train_now, _ = prepare_table(load_data(data_cfg["train_path"], tokens), config, clientes)
test_now, _ = prepare_table(load_data(data_cfg["test_path"], tokens), config, clientes)
history = pd.concat([train_now, test_now], ignore_index=True)
reports = []
for file, table in [("train.csv", train_now), ("test.csv", test_now)]:
    _, report = fill_from_customer_history(table, history, diag_cfg, data_cfg["id_column"])
    reports.append(report.groupby(["step", "column"])["values"].sum().rename(file))
display(pd.concat(reports, axis=1).fillna(0).astype(int))

consecutive purchases with an observed gap: 6669; gap = date difference on 100.0%
missing gaps: 8, of which the previous purchase is in the data: 8


train.csv  test.csv
step                                              column                                           
filled from the customer's last earlier purchase  payment_type                        176        30
                                                  salesperson                         169        28
                                                  transport_method                    173        30
                                                  warehouse                           182        29
filled from the customer's previous purchase date days_since_previous_purchase          8         0
recomputed after the history fill                 prior_mean_gap_days                   8         0

- **The gap equals the date difference on 100% of 6,669 consecutive purchases**, so this is an exact rule, not an estimate.
- **All 8 missing gaps** have their previous purchase in the data and are filled exactly. `prior_mean_gap_days` then follows from the identity for the same 8 rows.
- **Category values filled:** 700 in train.csv and 117 in test.csv.

Only **earlier** purchases are read. A purchase is never filled from a later one, and the target is never used (`tests/test_pipeline.py` → `test_history_fill_uses_the_customers_past_purchases_only`). In cross-validation, a validation row can be filled from an earlier training row of the same customer. That is the information available when a purchase happens, so it isn't leakage.

# <font color='#BFD72F' size=6>**4. Keeping imputed values within their row's limits**</font> <a class="anchor" id="bounds"></a>

The limits come from the definitions of the columns and are checked by the consistency rules (notebook 01, section 5.2):

| Column | Lower limit | Upper limit |
|---|---|---|
| `spend_365d` | `spend_180d` | `prior_total_spend` |
| `prior_max_purchase_value` | `prior_mean_purchase_value` | total − (count − 1) × min |
| `days_since_previous_purchase`, `prior_mean_gap_days` | — | `customer_tenure_days` |

`RowBoundedImputer` imputes as before, then moves an **imputed** value to the nearest limit when it falls outside. Observed values are never changed. Below: the classification development set, the median fitted on it, with and without the limits.

[Back to TOC](#toc)

In [6]:
data = prepare_task_data(config, "classification")
recipe = data["preprocessing_config"]
features, bounds = recipe["numeric_features"], recipe["imputation"]["row_bounds"]
X = data["X_dev"][features].astype("float64")
free = pd.DataFrame(RowBoundedImputer(_numeric_imputer(recipe["imputation"]), features, {}).fit(X).transform(X), columns=features, index=X.index)
bounded = pd.DataFrame(RowBoundedImputer(_numeric_imputer(recipe["imputation"]), features, bounds).fit(X).transform(X), columns=features, index=X.index)

rows = []
for column, limits in bounds.items():
    imputed = X[column].isna()
    for name, filled in [("median", free), ("median + row limits", bounded)]:
        low = filled.eval(str(limits["min"])) if "min" in limits else pd.Series(-np.inf, index=X.index)
        high = filled.eval(str(limits["max"])) if "max" in limits else pd.Series(np.inf, index=X.index)
        outside = imputed & ((filled[column] < low - 0.01) | (filled[column] > high + 0.01))
        rows.append({"column": column, "imputer": name, "imputed values": int(imputed.sum()), "outside the row's limits": int(outside.sum())})
display(pd.DataFrame(rows).pivot(index="column", columns="imputer", values="outside the row's limits")
        .join(pd.DataFrame(rows).groupby("column")["imputed values"].first()))
observed_changed = int((~np.isclose(bounded.where(X.notna()), X, equal_nan=True)).sum())
print("observed values changed by the limits:", observed_changed)

,median,median + row limits,imputed values
column,,,
days_since_previous_purchase,0,0,0
prior_max_purchase_value,9,0,35
prior_mean_gap_days,0,0,0
spend_365d,77,0,137


observed values changed by the limits: 0


- **Without the limits, the median produces impossible values:**
  - `spend_365d`: **77 of the 137** imputed values (56%) fall outside what their own row allows;
  - `prior_max_purchase_value`: **9 of the 35**.
- **With the limits:** 0 impossible values, and 0 observed values changed.
- **The gap columns** no longer reach the imputer at all (section 3.2), so their limit is a safeguard for new records.

# <font color='#BFD72F' size=6>**5. Does it help the models?**</font> <a class="anchor" id="models"></a>

The three changes are added one at a time and scored on the same 5 temporal folds with each task's chosen recipe and model (notebook 01, section 7):
- **A, previous pipeline;**
- **B, A + exact recoveries;**
- **C, B + customer history;**
- **D, C + row limits** (the current configuration).

[Back to TOC](#toc)

In [7]:
ablation = {"A previous": previous, "B + exact recoveries": variant(True, False, False),
            "C + customer history": variant(True, True, False), "D + row limits (current)": config}
ablation_scores = {}
for task in ["classification", "regression"]:
    metric = config[task]["selection_metric"]
    higher = metric != "rmse"
    scores = {}
    for name, cfg in ablation.items():
        d = prepare_task_data(cfg, task)
        folds = temporal_folds(d["extras_dev"][data_cfg["date_column"]], config["cv"]["n_splits"], config["cv"]["gap_days"])
        pipeline = build_pipeline(d["preprocessing_config"], TASKS[task].model.build_model(config[task]["model"]))
        scores[name] = cross_validate_pipeline(pipeline, d["X_dev"], d["y_dev"], folds, SCORING[task])[f"validation_{metric}"].to_numpy()
    ablation_scores[task] = scores
    names = list(scores)
    table = pd.DataFrame({f"mean {metric}": {k: v.mean() for k, v in scores.items()}, "std": {k: v.std(ddof=1) for k, v in scores.items()}})
    table["vs previous step"] = [""] + [versus(scores[names[i]], scores[names[i - 1]], higher) for i in range(1, len(names))]
    table["vs A"] = [""] + [versus(scores[n], scores[names[0]], higher) for n in names[1:]]
    print(f"\n{task} ({metric})")
    display(table.round(4))


classification (accuracy)


,mean accuracy,std,vs previous step,vs A
A previous,0.7071,0.0172,,
B + exact recoveries,0.7069,0.0169,"-0.0002 (SE 0.0006, better on 1/5 folds)","-0.0002 (SE 0.0006, better on 1/5 folds)"
C + customer history,0.7078,0.0152,"+0.0010 (SE 0.0012, better on 3/5 folds)","+0.0007 (SE 0.0016, better on 3/5 folds)"
D + row limits (current),0.7076,0.0156,"-0.0002 (SE 0.0002, better on 0/5 folds)","+0.0005 (SE 0.0016, better on 3/5 folds)"



regression (rmse)


,mean rmse,std,vs previous step,vs A
A previous,1943.2316,165.4157,,
B + exact recoveries,1944.4378,165.0719,"-1.2062 (SE 0.5729, better on 1/5 folds, decis...","-1.2062 (SE 0.5729, better on 1/5 folds, decis..."
C + customer history,1946.6769,168.8338,"-2.2391 (SE 1.8780, better on 1/5 folds)","-3.4453 (SE 1.8512, better on 1/5 folds)"
D + row limits (current),1949.2400,175.4222,"-2.5632 (SE 4.5983, better on 3/5 folds)","-6.0085 (SE 6.2261, better on 2/5 folds)"


- **Classification:** flat. The three changes together: +0.0005 accuracy, better on 3 of 5 folds, not decisive.
- **Regression:** each step is a few RMSE worse on the mean, out of a fold-to-fold spread of about ±170. Only the exact recoveries cross the 2-SE line (−1.2 RMSE), but they change one single cell. The seed check below tests whether that is real.

A random forest is itself random: two runs on almost identical data can differ by a few RMSE. To tell a real effect from that noise, the regression comparison A vs D is repeated with three other forest seeds.

In [8]:
task, metric = "regression", "rmse"
seed_rows = []
datasets = {name: prepare_task_data(cfg, task) for name, cfg in [("A", previous), ("B", variant(True, False, False)), ("D", config)]}
folds = temporal_folds(datasets["A"]["extras_dev"][data_cfg["date_column"]], config["cv"]["n_splits"], config["cv"]["gap_days"])
for seed in [1, 2, 3]:
    model_cfg = {**config[task]["model"], "params": {**config[task]["model"]["params"], "random_state": seed}}
    s = {name: cross_validate_pipeline(build_pipeline(d["preprocessing_config"], TASKS[task].model.build_model(model_cfg)),
                                       d["X_dev"], d["y_dev"], folds, SCORING[task])[f"validation_{metric}"].to_numpy()
         for name, d in datasets.items()}
    seed_rows.append({"forest seed": seed, "B vs A": versus(s["B"], s["A"], False), "D vs A": versus(s["D"], s["A"], False)})
display(pd.DataFrame(seed_rows))
changed = (datasets["A"]["X_dev"] != datasets["B"]["X_dev"]) & ~(datasets["A"]["X_dev"].isna() & datasets["B"]["X_dev"].isna())
print("development-set cells changed by the exact recoveries (regression):", int(changed.sum().sum()))

,forest seed,B vs A,D vs A
0,1,"-0.2702 (SE 0.1287, better on 1/5 folds, decis...","+0.3824 (SE 3.1946, better on 4/5 folds)"
1,2,"+0.1693 (SE 0.1455, better on 4/5 folds)","-6.7963 (SE 5.9429, better on 1/5 folds)"
2,3,"+0.4445 (SE 0.3666, better on 2/5 folds)","-4.2896 (SE 4.2390, better on 1/5 folds)"


development-set cells changed by the exact recoveries (regression): 1


**Across the four forest seeds (42, 1, 2, 3):**
- **the three changes together** move the regression RMSE by −6.0, +0.4, −6.8 and −4.3. That is about −4 on average (0.2% of the RMSE), and never decisive;
- **the exact recoveries** move it by −1.2, −0.3, +0.2 and +0.4. They change 1 cell of the regression development set, so their "decisive" result at seed 42 is forest noise, not an effect.

**Decision: keep all three changes.** By the rule of notebook 01 (change for a decisive gain, or for a clear reason that most folds confirm), neither model gains decisively and neither loses decisively. The reason to keep them is **correctness**, the same reason that kept the consistency rules in notebook 01, section 5.2 despite a small loss:
- values computed exactly, or taken from the customer's own past, instead of an overall median;
- no imputed value that the data's own rules forbid.

**A likely reason for the small regression cost:** its explicit `"Missing"` category marked purchases with incomplete records, and the history fill removes most of those gaps. A "filled from history" indicator would bring that signal back. It isn't added, because the cost isn't significant.

# <font color='#BFD72F' size=6>**6. Alternative imputers, again**</font> <a class="anchor" id="imputers"></a>

Notebook 01, section 12 compared the imputers when more gaps reached them. With fewer gaps and the row limits, the same comparison is repeated (each imputer fitted inside each fold; KNN and iterative impute in scaled space).

[Back to TOC](#toc)

In [9]:
imputer_tables = {}
for task in ["classification", "regression"]:
    d = prepare_task_data(config, task)
    folds = temporal_folds(d["extras_dev"][data_cfg["date_column"]], config["cv"]["n_splits"], config["cv"]["gap_days"])
    recipe = d["preprocessing_config"]
    current = recipe["imputation"]
    variants = {f"numeric {s}": {**recipe, "imputation": {**current, "numeric_strategy": s}}
                for s in ["median", "mean", "constant", "knn", "iterative"]}
    other_categorical = "most_frequent" if current["categorical_strategy"] == "constant" else "constant"
    variants[f"categorical {other_categorical}"] = {**recipe, "imputation": {**current, "categorical_strategy": other_categorical}}
    metric = config[task]["selection_metric"]
    table, per_fold = compare_preprocessing_variants(
        variants, lambda: TASKS[task].model.build_model(config[task]["model"]), d["X_dev"], d["y_dev"], folds, SCORING[task], metric)
    table["vs numeric median"] = ["" if v == "numeric median" else versus(per_fold[v], per_fold["numeric median"], metric != "rmse")
                                  for v in table["variant"]]
    imputer_tables[task] = table
    print(f"\n{task} (current: numeric {current['numeric_strategy']}, categorical {current['categorical_strategy']}): validation {metric}")
    display(table.round(4))


classification (current: numeric median, categorical most_frequent): validation accuracy


,variant,mean_accuracy,std_accuracy,vs numeric median
0,numeric iterative,0.7081,0.0157,"+0.0005 (SE 0.0005, better on 1/5 folds)"
1,numeric mean,0.7078,0.0155,"+0.0002 (SE 0.0002, better on 1/5 folds)"
2,numeric knn,0.7078,0.0161,"+0.0002 (SE 0.0012, better on 1/5 folds)"
3,numeric median,0.7076,0.0156,
4,categorical constant,0.7069,0.0179,"-0.0007 (SE 0.0013, better on 2/5 folds)"
5,numeric constant,0.7064,0.0165,"-0.0012 (SE 0.0007, better on 0/5 folds)"



regression (current: numeric median, categorical constant): validation rmse


,variant,mean_rmse,std_rmse,vs numeric median
0,numeric knn,1941.7155,169.4981,"+7.5245 (SE 4.7112, better on 4/5 folds)"
1,numeric iterative,1943.0779,162.9468,"+6.1622 (SE 6.9481, better on 4/5 folds)"
2,numeric constant,1945.9921,178.9548,"+3.2480 (SE 1.9809, better on 5/5 folds)"
3,numeric mean,1946.2947,167.3833,"+2.9453 (SE 4.8572, better on 4/5 folds)"
4,numeric median,1949.2400,175.4222,
5,categorical most_frequent,1951.6897,170.8223,"-2.4496 (SE 2.4099, better on 2/5 folds)"


- **Classification:** every imputer is within 0.0012 accuracy of the median, and none is decisive, so **the median stays**.
- **Regression:** KNN is still first (7.5 RMSE better, 4 of 5 folds) but **no longer decisive** (SE 4.7). Before these changes it was decisive: 6.0 RMSE, SE 2.9, all 5 folds (notebook 01, section 12). Fewer and better-filled gaps leave less for a clever imputer to do. By the rule, **the median stays**: it is simpler and faster. This closes the pending KNN decision.
- **`constant` (0)** is better on all 5 regression folds (3.2 RMSE, SE 2.0) but not decisive. In this pipeline 0 already means "not applicable" (notebook 01, section 2.6), so filling real gaps with it would merge two situations. Not adopted.
- **Categorical:** the regression keeps `"Missing"` (most frequent is 2.4 RMSE worse), and the classification keeps the most frequent value.

# <font color='#BFD72F' size=6>**7. Model scores with the final recipe**</font> <a class="anchor" id="scores"></a>

The candidate models of each task (`<task>.candidate_models`), scored with the final preprocessing. These numbers replace those of notebook 01, section 10 in the README.

[Back to TOC](#toc)

In [10]:
model_rows = []
for task in ["classification", "regression"]:
    d = prepare_task_data(config, task)
    folds = temporal_folds(d["extras_dev"][data_cfg["date_column"]], config["cv"]["n_splits"], config["cv"]["gap_days"])
    metric = config[task]["selection_metric"]
    for model_cfg in config[task]["candidate_models"]:
        s = cross_validate_pipeline(build_pipeline(d["preprocessing_config"], TASKS[task].model.build_model(model_cfg)),
                                    d["X_dev"], d["y_dev"], folds, SCORING[task])
        model_rows.append({"task": task, "model": model_cfg["type"], f"train": s[f"train_{metric}"].mean(),
                           "validation": s[f"validation_{metric}"].mean(), "validation std": s[f"validation_{metric}"].std(ddof=1)})
display(pd.DataFrame(model_rows).round(4))

,task,model,train,validation,validation std
0,classification,dummy,0.5532,0.5530,0.0345
1,classification,logistic_regression,0.7084,0.7076,0.0156
2,classification,decision_tree,0.7378,0.6515,0.0397
3,classification,random_forest,0.8508,0.7074,0.0156
4,regression,dummy,1940.1767,2471.5528,237.7549
5,regression,ridge,1484.7682,2052.3645,290.9243
6,regression,decision_tree,1208.9477,2438.0471,219.4229
7,regression,random_forest,985.9261,1949.2400,175.4222


These replace the numbers of notebook 01, section 10 in the README.

- **Classification:** logistic regression (0.7076) stays the chosen model. The random forest ties it (0.7074) and overfits (0.85 on training).
- **Regression:** the random forest (1,949) stays the chosen model, 103 RMSE ahead of ridge (2,052).
- **The depth-5 decision tree** moves the most between versions (0.65 / 2,438), as small trees do.

# <font color='#BFD72F' size=6>**8. Summary**</font> <a class="anchor" id="summary"></a>

| Change | Evidence | Gaps | Models (temporal CV) | Decision |
|---|---|---|---|---|
| Exact recoveries: largest / smallest previous purchase with 1–2 previous purchases (`recompute_rules`) | identities hold on 100% of 466 / 469 / 387 rows | 4 values recovered exactly | no effect (1 cell in the regression development set) | **kept** |
| Customer history: categories (`customer_history.last_observed_columns`) | the last value is right on 81–97% of purchases vs 17–73% for the most frequent value (warehouse: tie) | 700 values in train.csv, 117 in test.csv | — | **kept** |
| Customer history: days since the previous purchase (`customer_history.previous_purchase_gap`) | = date difference on 100% of 6,669 purchases | all 8 gaps, plus the 8 mean gaps | — | **kept** |
| Row limits on imputed values (`imputation.row_bounds`) | the median broke them on 77 / 137 `spend_365d` and 9 / 35 `prior_max_purchase_value` | 0 impossible imputed values, 0 observed values changed | — | **kept** |
| All three together | — | development set 792 → 249 (classification), 507 → 174 (regression) | classification +0.0005 (not decisive); regression −6 RMSE at seed 42, about −4 over 4 seeds (never decisive) | **kept: correctness** |
| KNN instead of the median (regression) | +7.5 RMSE, SE 4.7, 4 of 5 folds: no longer decisive | — | — | **median stays** |

**What it doesn't change:** the 137 `spend_365d` gaps of customers with more than a year of history (median + row limits + missing indicator), and categories on a customer's first known purchase (most frequent / `"Missing"`).